In [ ]:
import time
start_time = time.time()
import pandas as pd
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
import spark_sdk as ss
import yaml
from dateutil.relativedelta import relativedelta
from pyspark.sql.functions import col, sum as sum_, count, max as max_, datediff, lit, min as min_, desc, when, unix_timestamp, row_number, struct, collect_list,\
to_date, countDistinct, exp, lit
from pyspark.sql.window import Window
from datetime import datetime, timedelta
import pytz

import spark_sdk as ss
ss.PySpark(yarn=False, num_executors=48, driver_memory='32g', executor_memory='32g',
           add_on_config0=('spark.port.maxRetries', '1000'),
           add_on_config1=("spark.sql.catalog.spark_catalog","org.apache.spark.sql.delta.catalog.DeltaCatalog"),
           add_on_config2=('spark.databricks.delta.retentionDurationCheck.enabled', 'false'),
           add_on_config3=('spark.sql.sources.partitionOverwriteMode', 'dynamic')
          )
spark = ss.PySpark().spark

from pyspark.sql.functions import to_date

full_dataset = spark.read.format('parquet').load('file:/home/baond12/projects/AI_Recommend_System_2025/BTL_HTTM/CS-recommendation-system/backend/full_dataset.parquet')\
.withColumn('event_date', to_date('TimePlaced'))

# Lấy danh sách 7 ngày cuối cùng có trong dữ liệu
max_date_row = full_dataset.agg(max_("event_date")).collect()[0][0]
# Tính toán ngày bắt đầu validation (cách max date 6 ngày)
val_date = max_date_row - timedelta(days=5)

start_train_date = val_date - timedelta(days=60)

end_train_date = val_date - timedelta(days=1)

print(f"Start Training Date: {start_train_date} -- End Training Date: {end_train_date}")
print(f"Validation Date: {val_date}")

train_data = full_dataset.filter(f"event_date <= '{end_train_date}'")
val_data = full_dataset.filter(f"event_date = '{val_date}'")
new_user_df = val_data.join(train_data, ['Account_Id'], 'left_anti').agg(countDistinct('Account_Id').alias('nb_new_users'))

print(f"Số lượng khách hàng mới: {new_user_df.collect()[0][0]}")

def item_feature_engineering(data_source):
    max_event_time_per_user = data_source.groupBy('Account_Id').agg(max_('event_date').alias('max_event_time_per_user'))
    ds_with_max_time = data_source.join(max_event_time_per_user, ['Account_Id'], 'inner')\
.select('event_date', 'Account_Id', 'ProductId', 'max_event_time_per_user')
    ds_with_max_time = ds_with_max_time.withColumn(
    "days_diff_to_max", datediff(col("max_event_time_per_user"), col("event_date"))
)

    lambda_decay = 0.05
    ds_with_max_time = ds_with_max_time.withColumn(
        "decayed_score", exp(-lit(lambda_decay) * col("days_diff_to_max"))
    )
    
    ds_rating = ds_with_max_time.groupBy("Account_Id", "ProductId")\
        .agg({"decayed_score": "sum"})\
        .withColumnRenamed("sum(decayed_score)", "importance_score")
    
    return ds_rating

train_feature_engineering = item_feature_engineering(train_data)

user_item_feature = train_data.groupBy('Account_Id', 'ProductId').agg(
count('*').alias('nb_purchase'),
sum_('Price').alias('revenue'))

# train_feature_engineering.show(2)
# user_item_feature.show(2)

def normalize_score(df, score_col, group_cols=['Account_Id']):
    stats = df.groupBy(*group_cols).agg(
        min_(score_col).alias("min_score"),
        max_(score_col).alias("max_score")
    )
    df = df.join(stats, on=group_cols, how="left")
    df = df.withColumn(
        f"{score_col}_norm",
        when(col("max_score") != col("min_score"),
             (col(score_col) - col("min_score")) / (col("max_score") - col("min_score"))
        ).otherwise(lit(0))
    ).drop("min_score", "max_score", score_col)
    df = df.withColumnRenamed(f"{score_col}_norm", score_col)
    return df

user_item_feature_norm = normalize_score(user_item_feature, 'nb_purchase', group_cols=['Account_Id'])
user_item_feature_norm = normalize_score(user_item_feature_norm, 'revenue', group_cols=['Account_Id'])

from pyspark.sql.functions import udf
from pyspark.sql.types import DoubleType
from scipy.stats import norm

norm_cdf_udf = udf(lambda z: float(norm.cdf(z)), DoubleType())

user_item_feature_norm = user_item_feature_norm.withColumn("nb_purchase_scale", norm_cdf_udf(col("nb_purchase")))
user_item_feature_norm = user_item_feature_norm.withColumn("revenue_scale", norm_cdf_udf(col("revenue")))

# user_item_feature_norm.show(2)

user_item_feature_norm = user_item_feature_norm.withColumn(
    "final_score",
    0.65 * col("revenue_scale") + 0.35 * col("nb_purchase_scale")
).select('Account_Id', 'ProductId', 'final_score')

concat_features = user_item_feature_norm.join(train_feature_engineering, ['Account_Id', 'ProductId'], 'outer')\
.fillna(0.0001, subset=['final_score', 'importance_score']).withColumn('item_pred_score', col('final_score')*col('importance_score'))\
.select('Account_Id', 'ProductId', 'item_pred_score')

concat_features = concat_features.withColumn('d', lit(val_date.strftime('%Y-%m-%d')))
concat_features.write.format('parquet')\
.mode('overwrite').option('overwriteSchema', 'true')\
.partitionBy(['d'])\
.save("file:/mnt/projects-data/recommendation/dev_folder/baond12/btl_httm/user_item_recs_final_local.parquet")